## Sensitivity Analysis of W&B Runs

In [1]:
import wandb

# Connect to Wandb
wandb.login()

# Specify the project name
project_name = "karman"

# Specify the date after which you want to retrieve the run IDs
start_date = "2024-08-04T19:12:00"      # August 4th, 2024 at 8:12 pm

# Get the runs for the specified project and date
api = wandb.Api()

runs = api.runs(path="karman", filters={"createdAt": {"$gte": start_date}}) #, created_after=start_date)

# Extract the run IDs
run_ids = [run.id for run in runs]

# Print the run IDs
print(run_ids)

# Check these are all past start_date, will error if not
assert(all([r.createdAt > start_date for r in runs]))

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: fdlteam8 (trilliumtechnologies). Use `wandb login --relogin` to force relogin


['8g0pysdr', '5ohv0f15', 'iok3557p', '1ct7n7yy', 'uthb5v1v', 'p4xe3b2r', 'ts9xs3wo', 'zb61zyhg', 'zqqm7ned', 'r95l506l', 'xcour2n0', 'ba0elg1q', '8sv64aqn', 'i45e8sw0', 'x76t0p07', 'a0lz32if', 'szgfk166', 'm0zvxq1e', 'wdlf5xfp', 'r6ht7q9h', 'sxp477yk', '08w0skva', '91ih08zk']


In [2]:
runs[0].history(samples=3)

wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.
wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.


CommError: HTTPSConnectionPool(host='api.wandb.ai', port=443): Read timed out. (read timeout=19)

In [4]:
cols = runs[0].history(samples=10).columns
cols

Index(['nn_mse_train', 'nrlmsise00_mape_train', '_step', '_runtime',
       'nrlmsise00_mse_train', 'nn_mape_train', 'q_loss_train', '_timestamp',
       'nrlmsise00_mape_train_total', 'nrlmsise00_mse_train_total',
       'nn_mape_train_total', 'q_loss_train_total', 'nn_mse_train_total',
       'nn_mape_valid', 'nrlmsise00_mape_valid', 'nn_mse_valid',
       'q_loss_valid', 'nrlmsise00_mse_valid', 'nrlmsise00_mse_valid_total',
       'nrlmsise00_mape_valid_total', 'nn_mape_valid_total',
       'q_loss_valid_total', 'nn_mse_valid_total'],
      dtype='object')

In [5]:
"nn_mape_valid" in cols

True

In [2]:
# nn_mape_train, nn_mape_valid

run = runs[0]

In [18]:
keys

['nn_mape_valid_total',
 'nrlmsise00_mse_valid_total',
 'q_loss_valid_total',
 'q_loss_train_total',
 'nn_mape_train_total',
 'nrlmsise00_mape_train_total',
 'nrlmsise00_mape_valid_total',
 'nn_mse_train_total',
 'nn_mse_valid_total',
 'nrlmsise00_mse_train_total']

In [35]:
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
valid_history = run.history(keys=["_step"]+valid_keys) 

train_keys = [k for k in run.summary.keys() if "train_total" in k]
train_history = run.history(keys=["_step"]+train_keys) 

# for i in history:
#     if i["_step"] == 100:
#         print(i["nn_mape_train"])#, i["nn_mape_valid"])
#         break
valid_history

,_step,nn_mape_valid_total,nrlmsise00_mse_valid_total,q_loss_valid_total,nrlmsise00_mape_valid_total,nn_mse_valid_total
0,28201,27.356451,0.008957,0.207591,48.823883,0.004196
1,56403,22.608013,0.008957,0.209701,48.823826,0.003540
2,84605,21.390541,0.008956,0.211569,48.820110,0.003651
3,112807,22.944304,0.008956,0.211033,48.820095,0.003780
4,141009,22.401928,0.008958,0.211877,48.826431,0.003849
5,169211,23.735941,0.008956,0.211410,48.819786,0.003924
6,197413,23.234953,0.008957,0.210931,48.825012,0.003819
7,225615,23.214739,0.008957,0.211591,48.821999,0.003878
8,253817,23.732193,0.008958,0.209346,48.826084,0.003975


In [42]:
for c in train_history.columns:
    if c == "_step":
        continue
    print(c, train_history[c].min(), train_history[c].argmin())
    # print(train_history[c][], train_history[c].argmax())

_step 25654 0
q_loss_train_total 0.19893452633816147 0
nn_mape_train_total 18.062211990356445 8
nrlmsise00_mape_train_total 47.16281509399414 2
nn_mse_train_total 0.0024647616039170494 8
nrlmsise00_mse_train_total 0.00835809763520956 3


In [25]:
import pandas as pd

run = runs[0]
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
train_keys = [k for k in run.summary.keys() if "train_total" in k]

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

results = []

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    # Should really do this with argmin on one and take the values from the rest!!
    train_arg_min = train_history["nn_mape_train_total"].argmin()
    valid_arg_min = valid_history["nn_mape_valid_total"].argmin()
    results.append({'Run ID': run.id, "Run Name": run.name.replace("run_gpu", "").replace("_epochs_"+str(run.config["epochs"]), ""), "Lag": run.config["lag_minutes"], "Epochs": run.config["epochs"],
                                    **{key: train_history[key][train_arg_min] for key in train_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]},
                                    **{key: valid_history[key][valid_arg_min] for key in valid_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]}
    })

    # results_df = pd.concat([results_df, pd.DataFrame({'Run ID': run.id, 
    #                                 **{key: train_history[key].min() for key in train_keys if key != "_step"},
    #                                 **{key: valid_history[key].min() for key in valid_keys if key != "_step"}
    # })], ignore_index=True)

# Print the results
print(results_df)
pd.DataFrame(results).sort_values("nn_mape_valid_total")

Empty DataFrame
Columns: [Run ID, q_loss_train_total, nrlmsise00_mape_train_total, nn_mape_train_total, nn_mse_train_total, nrlmsise00_mse_train_total, nn_mape_valid_total, nrlmsise00_mse_valid_total, nrlmsise00_mape_valid_total, nn_mse_valid_total, q_loss_valid_total]
Index: []


,Run ID,Run Name,Lag,Epochs,nrlmsise00_mape_train_total,nn_mape_train_total,nn_mse_train_total,nrlmsise00_mse_train_total,nn_mape_valid_total,nrlmsise00_mse_valid_total,nrlmsise00_mape_valid_total,nn_mse_valid_total
16,szgfk166,run_tft_first_w_soho_and_w_omni_and_proxies,500,20,47.163149,11.760783,0.001111,0.008358,13.100392,0.008957,48.822879,0.001429
6,ts9xs3wo,run_tft_everything,500,20,47.163406,11.861155,0.001133,0.008358,13.719629,0.008957,48.825405,0.001545
18,wdlf5xfp,_tft_wo_sw_all_except_s107_30_epochs,500,30,47.163498,11.997643,0.001142,0.008358,14.292094,0.008957,48.824821,0.001614
3,1ct7n7yy,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,60000,20,47.163177,12.610574,0.001295,0.008358,14.299049,0.008957,48.823654,0.001642
15,a0lz32if,run_tft_wo_omni_except_omni_indices_and_wo_ap_...,500,20,47.163284,12.639576,0.001264,0.008358,14.408853,0.008957,48.825375,0.001637
14,x76t0p07,run_tft_wo_omni_except_omni_magnetic_field_and...,500,20,47.163361,12.650912,0.001262,0.008358,15.247732,0.008957,48.823292,0.001981
4,uthb5v1v,_tft_w_omni_and_soho_wo_indices_and_proxies,500,30,47.163334,11.612515,0.001085,0.008358,15.306245,0.008958,48.826031,0.001948
19,r6ht7q9h,_tft_wo_sw_all_except_m107_30_epochs,500,30,47.162983,12.273036,0.001196,0.008358,15.873594,0.008956,48.821018,0.001905
13,i45e8sw0,run_tft_wo_omni_except_omni_solar_wind_and_wo_...,500,20,47.163792,13.060397,0.001349,0.008358,16.648859,0.008956,48.818928,0.002318
17,m0zvxq1e,_tft_wo_sw_all_except_y107_30_epochs,500,30,47.163040,11.972352,0.001139,0.008358,18.866863,0.008956,48.819500,0.002648


In [20]:
run.config

{'lr': 0.001,
 'device': 'cuda:0',
 'epochs': 30,
 'dropout': 0.05,
 'max_date': '2024-05-31 23:59:32',
 'min_date': '2000-07-29 00:59:47',
 'run_name': 'run_gpu_tft_soho_only_w_40000_lag_100_resolution',
 'goes_path': None,
 'soho_path': '../data/soho_data/soho_data.csv',
 'batch_size': 64,
 'model_path': None,
 'state_size': 64,
 'torch_type': 'float32',
 'lag_minutes': 40000,
 'lstm_layers': 2,
 'num_workers': 16,
 'thermo_path': '../data/satellites_data_w_sw_2mln.csv',
 'wandb_inactive': False,
 'attention_heads': 4,
 'nrlmsise00_path': '../data/nrlmsise00_data/nrlmsise00_time_series.csv',
 'omni_indices_path': 'None',
 'resolution_minutes': 100,
 'omni_solar_wind_path': 'None',
 'normalization_dict_path': None,
 'omni_magnetic_field_path': 'None',
 'features_to_exclude_thermo': 'celestrack__ap_average__,JB08__d_st_dt__[K],space_environment_technologies__f107_obs__,space_environment_technologies__f107_average__,space_environment_technologies__s107_obs__,space_environment_technologi

In [19]:
runs[0].config['lag_minutes']

500

In [ ]:
import pandas as pd

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    results_df = results_df.append({'Run ID': run.id, 
                                    **{key: train_history[key].min() for key in train_keys if key not "_step"},
                                    **{key: valid_history[key].min() for key in valid_keys if ket not "_step"}
    }, ignore_index=True)

# Print the results
print(results_df)

In [29]:
history = run.scan_history(keys=["_step", "nn_mape_valid"])
# for i in history:
#     if i["_step"] == 100:
#         print(i["nn_mape_train"])#, i["nn_mape_valid"])
#         break
history.shape

AttributeError: 'SampledHistoryScan' object has no attribute 'shape'

In [28]:
history.shape

AttributeError: 'SampledHistoryScan' object has no attribute 'shape'

In [12]:
[k for k in runs[0].summary.keys() if "_total" in k]

['nn_mape_valid_total',
 'nrlmsise00_mse_valid_total',
 'q_loss_valid_total',
 'q_loss_train_total',
 'nn_mape_train_total',
 'nrlmsise00_mape_train_total',
 'nrlmsise00_mape_valid_total',
 'nn_mse_train_total',
 'nn_mse_valid_total',
 'nrlmsise00_mse_train_total']

In [11]:
run.summary["nn_mape_valid_total"]


23.732192993164062